# 📚 e-Library: AI-Powered Book Recommendation System
### DS 2.2 G17 — Complete AIML Implementation Guide

---

## Architecture Overview

```
book__1_.csv
     │
     ▼
┌─────────────────────┐
│ 1. Data Preprocessing│  ← Clean & structure metadata
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 2. User Query        │  ← Parse natural language input
│    Processing        │
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 3. Semantic Similarity│ ← TF-IDF + Sentence-BERT embeddings
│    Logic             │
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 4. Search Integration│ ← Combined keyword + semantic search
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 5. Testing &         │ ← Precision, Recall, NDCG metrics
│    Evaluation        │
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 6. Ranking & Filtering│ ← Score fusion + genre/rating filters
└────────┬────────────┘
         ▼
┌─────────────────────┐
│ 7. Feedback Data     │ ← Collect & retrain on user feedback
│    Handling          │
└─────────────────────┘
```

---

## Models Used

| Component | Model / Technique | Why |
|---|---|---|
| Text vectorization (fast) | **TF-IDF** (sklearn) | Lightweight, no GPU needed |
| Semantic embeddings | **all-MiniLM-L6-v2** (Sentence-BERT) | Best speed/accuracy ratio for small datasets |
| Similarity search | **Cosine Similarity** | Standard for text vectors |
| Keyword extraction | **KeyBERT** | Pulls key phrases from queries |
| Feedback retraining | **Incremental TF-IDF + SVD** | No full retrain needed |

---

## 🔧 STEP 0: Install Dependencies
Run this cell **once** before anything else.

In [1]:
# ============================================================
# STEP 0: Install all required packages
# Run this cell once. Restart kernel after installation.
# ============================================================
import subprocess, sys

packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "sentence-transformers",  # Sentence-BERT (all-MiniLM-L6-v2)
    "keybert",               # Keyword extraction from queries
    "nltk",                  # Text preprocessing (stopwords, tokenization)
    "matplotlib",
    "seaborn",
    "tqdm",                  # Progress bars
    "joblib",                # Save/load models
    "scipy",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages installed. Restart kernel now if this is your first run.")

✅ All packages installed. Restart kernel now if this is your first run.


In [2]:
# Download NLTK data (run once)
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
print("✅ NLTK data downloaded.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...


✅ NLTK data downloaded.


---
# 🧹 STEP 1: Data Preprocessing
**Goal:** Clean and structure book metadata from `book__1_.csv` so it is ready for AI model input.

**What this does:**
- Loads and inspects the dataset
- Handles missing values
- Cleans text (remove HTML, special characters, lowercase)
- Builds a combined `content` field: `title + authors + description`
- Saves the cleaned dataset

In [3]:
# ============================================================
# STEP 1: DATA PREPROCESSING
# ============================================================
import pandas as pd
import numpy as np
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ----- 1.1 Load dataset -----
# ⚠️ Update this path to your actual CSV location
CSV_PATH = "book__1_.csv"  # <-- change if needed

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

print("📊 Dataset Shape:", df.shape)
print("\n📋 Columns:", df.columns.tolist())
print("\n🔍 First row preview:")
df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'book__1_.csv'

In [ ]:
# ----- 1.2 Inspect missing values -----
print("❓ Missing values per column:")
print(df.isnull().sum())

# ----- 1.3 Handle missing values -----
df['description'] = df['description'].fillna('')
df['authors']     = df['authors'].fillna('Unknown Author')
df['language_code'] = df['language_code'].fillna('eng')
df['average_rating'] = pd.to_numeric(df['average_rating'], errors='coerce').fillna(3.5)
df['original_publication_year'] = pd.to_numeric(
    df['original_publication_year'], errors='coerce'
).fillna(0).astype(int)

# Drop unnamed index column if present
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print("\n✅ Missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# ----- 1.4 Text Cleaning Function -----
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Full text cleaning pipeline:
    1. Remove HTML tags
    2. Lowercase
    3. Remove punctuation & numbers
    4. Remove stopwords
    5. Lemmatize words
    """
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)          # Remove HTML
    text = text.lower()                            # Lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)         # Keep only letters
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

# Apply to each text field
print("⏳ Cleaning text fields...")
df['clean_title']       = df['title'].apply(clean_text)
df['clean_authors']     = df['authors'].apply(clean_text)
df['clean_description'] = df['description'].apply(clean_text)

# ----- 1.5 Build combined content field -----
# Title gets 3x weight, authors 2x, description 1x
df['content'] = (
    df['clean_title'] + ' ' + df['clean_title'] + ' ' + df['clean_title'] + ' ' +
    df['clean_authors'] + ' ' + df['clean_authors'] + ' ' +
    df['clean_description']
)

# Remove empty content rows
df = df[df['content'].str.strip() != ''].reset_index(drop=True)

print(f"\n✅ Cleaned dataset: {len(df)} books ready for AI processing")
print("\n📖 Sample content field:")
print(df['content'].iloc[0][:300])

In [ ]:
# ----- 1.6 Save cleaned dataset -----
df.to_csv('books_cleaned.csv', index=False)
print("💾 Saved: books_cleaned.csv")

# Quick statistics
print(f"\n📊 Dataset Summary:")
print(f"  Total books:         {len(df)}")
print(f"  Avg rating:          {df['average_rating'].mean():.2f}")
print(f"  Books with desc:     {(df['description'].str.len() > 10).sum()}")
print(f"  Avg description len: {df['description'].str.len().mean():.0f} chars")

---
# 💬 STEP 2: User Query Processing
**Goal:** Take a user's natural language query (e.g. *"exciting adventure book for teens"*) and transform it into structured intent for the recommendation engine.

**What this does:**
- Cleans and normalizes the query
- Extracts key phrases using **KeyBERT**
- Detects intent: mood, genre, author, year range
- Returns a structured query object for the next step

In [ ]:
# ============================================================
# STEP 2: USER QUERY PROCESSING
# ============================================================
from keybert import KeyBERT
import re

# Initialize KeyBERT model (downloads ~90MB on first run)
print("⏳ Loading KeyBERT model...")
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print("✅ KeyBERT ready")

# ----- 2.1 Intent / Filter Detector -----
MOOD_KEYWORDS = {
    'happy':       ['happy', 'funny', 'comedy', 'lighthearted', 'humorous', 'cheerful'],
    'sad':         ['sad', 'emotional', 'tearful', 'melancholic', 'heartbreaking'],
    'adventurous': ['adventure', 'exciting', 'thrilling', 'action', 'quest', 'journey'],
    'romantic':    ['romance', 'love', 'relationship', 'romantic', 'passion'],
    'mysterious':  ['mystery', 'detective', 'suspense', 'thriller', 'crime', 'dark'],
    'educational': ['learn', 'educational', 'informative', 'factual', 'history', 'science'],
    'inspirational': ['inspiring', 'motivational', 'self-help', 'growth', 'empowering'],
}

def detect_intent(query: str) -> dict:
    """
    Parses a user query into structured intent:
    - mood: detected mood/genre
    - year_range: (min_year, max_year) if mentioned
    - min_rating: minimum rating if mentioned
    - author_hint: author name if mentioned
    """
    q_lower = query.lower()
    intent = {
        'original_query': query,
        'mood': None,
        'year_range': None,
        'min_rating': None,
        'author_hint': None,
    }

    # Mood detection
    for mood, keywords in MOOD_KEYWORDS.items():
        if any(kw in q_lower for kw in keywords):
            intent['mood'] = mood
            break

    # Year range detection: "books from 1990s" or "2000 to 2010"
    year_match = re.search(r'(\d{4})\s*(?:to|-|–)\s*(\d{4})', q_lower)
    if year_match:
        intent['year_range'] = (int(year_match.group(1)), int(year_match.group(2)))
    else:
        decade_match = re.search(r'(\d{4})s', q_lower)
        if decade_match:
            y = int(decade_match.group(1))
            intent['year_range'] = (y, y + 9)

    # Rating hint: "highly rated", "rating above 4"
    if any(w in q_lower for w in ['top rated', 'highest rated', 'best rated', 'popular']):
        intent['min_rating'] = 4.0
    rating_match = re.search(r'rating\s*(?:above|over|>)\s*([0-9.]+)', q_lower)
    if rating_match:
        intent['min_rating'] = float(rating_match.group(1))

    # Author hint: "by [Author Name]"
    author_match = re.search(r'by\s+([A-Z][a-z]+(\s+[A-Z][a-z]+)+)', query)
    if author_match:
        intent['author_hint'] = author_match.group(1)

    return intent


def process_query(raw_query: str) -> dict:
    """
    Full query processing pipeline.
    Returns: {
        'clean_query': str,
        'keywords': list of (keyword, score),
        'intent': dict
    }
    """
    # Clean query
    clean_q = clean_text(raw_query)  # Reuse function from Step 1

    # Extract top keywords
    keywords = kw_model.extract_keywords(
        raw_query,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=5
    )

    # Detect intent
    intent = detect_intent(raw_query)

    return {
        'original_query': raw_query,
        'clean_query':    clean_q,
        'keywords':       keywords,
        'intent':         intent
    }


# ----- 2.2 Test query processing -----
test_queries = [
    "exciting adventure books for teenagers",
    "top rated romance novels from the 1990s",
    "mystery thriller with a female detective",
    "inspiring self-help book by Malcolm Gladwell",
]

for q in test_queries:
    result = process_query(q)
    print(f"\n🔍 Query: '{q}'")
    print(f"   Clean:    {result['clean_query']}")
    print(f"   Keywords: {[kw[0] for kw in result['keywords']]}")
    print(f"   Intent:   {result['intent']}")

---
# 🧠 STEP 3: Semantic Similarity Logic
**Goal:** Build TWO complementary similarity engines:
1. **TF-IDF** — fast keyword-level matching (no GPU)
2. **Sentence-BERT (all-MiniLM-L6-v2)** — deep semantic understanding

**Model choice rationale:**
- `all-MiniLM-L6-v2` is 5x faster than `all-mpnet-base-v2` with ~96% of the accuracy
- TF-IDF handles exact term matches that BERT can miss
- Both scores are later fused in Step 4

In [ ]:
# ============================================================
# STEP 3A: TF-IDF VECTORIZATION
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# ----- 3.1 Build TF-IDF matrix -----
print("⏳ Building TF-IDF matrix...")

tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,    # Vocabulary cap
    ngram_range=(1, 2),    # Unigrams + bigrams
    min_df=2,              # Word must appear in ≥2 books
    max_df=0.85,           # Ignore words in >85% of books
    sublinear_tf=True,     # Apply log normalization
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['content'])

print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"   → {tfidf_matrix.shape[0]} books × {tfidf_matrix.shape[1]} features")

# Save vectorizer for later use (API / web integration)
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix,     'tfidf_matrix.pkl')
print("💾 Saved: tfidf_vectorizer.pkl, tfidf_matrix.pkl")


def get_tfidf_recommendations(query: str, top_n: int = 10) -> pd.DataFrame:
    """
    Returns top_n books by TF-IDF cosine similarity to query.
    """
    clean_q = clean_text(query)
    q_vec   = tfidf_vectorizer.transform([clean_q])
    scores  = cosine_similarity(q_vec, tfidf_matrix).flatten()

    top_idx = scores.argsort()[::-1][:top_n]
    result  = df.iloc[top_idx][['title', 'authors', 'average_rating',
                                 'original_publication_year', 'description']].copy()
    result['tfidf_score'] = scores[top_idx].round(4)
    return result.reset_index(drop=True)


# Quick test
print("\n🔍 TF-IDF test — 'adventure journey quest'")
get_tfidf_recommendations('adventure journey quest', top_n=5)[['title', 'authors', 'tfidf_score']]

In [ ]:
# ============================================================
# STEP 3B: SENTENCE-BERT SEMANTIC EMBEDDINGS
# ============================================================
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

# ⚠️ First run downloads ~90MB model. Subsequent runs use cache.
print("⏳ Loading Sentence-BERT model: all-MiniLM-L6-v2")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

# ----- 3.2 Encode all book descriptions -----
# Uses the raw (not over-cleaned) description for richer semantics
descriptions = df['title'].fillna('') + '. ' + df['description'].fillna('')

print(f"\n⏳ Encoding {len(descriptions)} books with SBERT...")
print("   (This takes ~2-5 minutes. Embeddings are saved to disk after.)")

book_embeddings = sbert_model.encode(
    descriptions.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings shape: {book_embeddings.shape}")
print(f"   → {book_embeddings.shape[0]} books × {book_embeddings.shape[1]} dimensions")

# Save embeddings (so you never have to re-encode)
np.save('book_embeddings.npy', book_embeddings)
print("💾 Saved: book_embeddings.npy")


def get_sbert_recommendations(query: str, top_n: int = 10) -> pd.DataFrame:
    """
    Returns top_n books by Sentence-BERT cosine similarity to query.
    """
    q_embedding = sbert_model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_embedding, book_embeddings).flatten()

    top_idx = scores.argsort()[::-1][:top_n]
    result  = df.iloc[top_idx][['title', 'authors', 'average_rating',
                                 'original_publication_year', 'description']].copy()
    result['sbert_score'] = scores[top_idx].round(4)
    return result.reset_index(drop=True)


# Quick test
print("\n🔍 SBERT test — 'boy discovers magical school'")
get_sbert_recommendations('boy discovers magical school', top_n=5)[['title', 'authors', 'sbert_score']]

In [ ]:
# ============================================================
# STEP 3C: Load saved embeddings (for subsequent runs)
# ============================================================
# Run this cell instead of 3B on subsequent Jupyter sessions
# (no need to re-encode every time)

# book_embeddings = np.load('book_embeddings.npy')
# tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
# tfidf_matrix     = joblib.load('tfidf_matrix.pkl')
# print("✅ Loaded saved embeddings and TF-IDF matrix")

print("ℹ️  Uncomment the lines above to load from disk on future runs.")

---
# 🔎 STEP 4: Search Integration
**Goal:** Combine TF-IDF and SBERT scores into one unified search pipeline with a configurable fusion weight.

**Formula:**
```
final_score = α × sbert_score + (1-α) × tfidf_score
```
Default α = 0.6 (favour semantics over keywords)

In [ ]:
# ============================================================
# STEP 4: SEARCH INTEGRATION
# ============================================================
from sklearn.preprocessing import MinMaxScaler

def hybrid_search(
    query:      str,
    top_n:      int   = 10,
    alpha:      float = 0.6,   # Weight for SBERT (1-alpha → TF-IDF)
    min_rating: float = None,  # Optional: only books above this rating
    year_range: tuple = None,  # Optional: (min_year, max_year)
    author_hint:str   = None,  # Optional: author name substring
) -> pd.DataFrame:
    """
    Unified AI-powered book search combining:
      - Sentence-BERT semantic similarity
      - TF-IDF keyword similarity
      - Metadata filters (rating, year, author)

    Returns top_n books ranked by fused score.
    """
    # --- 4.1 Get raw similarity scores ---
    clean_q     = clean_text(query)
    q_tfidf_vec = tfidf_vectorizer.transform([clean_q])
    tfidf_scores = cosine_similarity(q_tfidf_vec, tfidf_matrix).flatten()

    q_sbert_vec = sbert_model.encode([query], convert_to_numpy=True)
    sbert_scores = cosine_similarity(q_sbert_vec, book_embeddings).flatten()

    # --- 4.2 Normalise both score arrays to [0,1] ---
    def norm(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)

    tfidf_norm = norm(tfidf_scores)
    sbert_norm = norm(sbert_scores)

    # --- 4.3 Fuse scores ---
    fused = alpha * sbert_norm + (1 - alpha) * tfidf_norm

    # --- 4.4 Apply metadata filters ---
    mask = np.ones(len(df), dtype=bool)  # Start: include all

    if min_rating is not None:
        mask &= (df['average_rating'] >= min_rating).values

    if year_range is not None:
        yr = df['original_publication_year']
        mask &= ((yr >= year_range[0]) & (yr <= year_range[1])).values

    if author_hint is not None:
        mask &= df['authors'].str.contains(author_hint, case=False, na=False).values

    fused[~mask] = -1  # Exclude filtered books

    # --- 4.5 Return top results ---
    top_idx = fused.argsort()[::-1][:top_n]
    result  = df.iloc[top_idx][[
        'book_id', 'title', 'authors', 'average_rating',
        'original_publication_year', 'image_url', 'description'
    ]].copy()
    result['tfidf_score'] = tfidf_norm[top_idx].round(4)
    result['sbert_score'] = sbert_norm[top_idx].round(4)
    result['final_score'] = fused[top_idx].round(4)

    return result.reset_index(drop=True)


# ----- 4.2 Demo searches -----
print("="*60)
print("📚 DEMO 1: 'adventure books for young adults'")
r1 = hybrid_search('adventure books for young adults', top_n=5)
print(r1[['title', 'authors', 'average_rating', 'final_score']].to_string(index=False))

print("\n" + "="*60)
print("📚 DEMO 2: 'romantic drama highly rated' (rating ≥ 4.0)")
r2 = hybrid_search('romantic drama', top_n=5, min_rating=4.0)
print(r2[['title', 'authors', 'average_rating', 'final_score']].to_string(index=False))

print("\n" + "="*60)
print("📚 DEMO 3: 'magic wizard school' (year 1990–2010)")
r3 = hybrid_search('magic wizard school', top_n=5, year_range=(1990, 2010))
print(r3[['title', 'authors', 'original_publication_year', 'final_score']].to_string(index=False))

---
# 📊 STEP 5: Testing & Evaluation
**Goal:** Measure how accurate and relevant the AI recommendations are.

**Metrics used:**
| Metric | What it measures |
|---|---|
| **Precision@K** | Of top K results, how many are actually relevant? |
| **Recall@K** | Of all relevant books, how many were found? |
| **NDCG@K** | Are the best matches ranked highest? (Normalized Discounted Cumulative Gain) |
| **MRR** | Mean Reciprocal Rank — how quickly is the first correct result found? |
| **Coverage** | What % of all books can ever be recommended? |

In [ ]:
# ============================================================
# STEP 5: TESTING & EVALUATION
# ============================================================
from sklearn.metrics import ndcg_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ----- 5.1 Create evaluation test set -----
# These are hand-labelled query → relevant book pairs for testing.
# In production, replace with real user click data.
TEST_CASES = [
    {
        'query': 'hunger games dystopian survival',
        'relevant_titles': ['The Hunger Games', 'Divergent', 'Maze Runner']
    },
    {
        'query': 'wizard school magic adventure',
        'relevant_titles': ['Harry Potter and the Sorcerer\'s Stone',
                            'Harry Potter and the Chamber of Secrets',
                            "The Magician's Nephew"]
    },
    {
        'query': 'teenage vampire romance',
        'relevant_titles': ['Twilight', 'New Moon', 'Eclipse']
    },
    {
        'query': 'classic southern american racism injustice',
        'relevant_titles': ['To Kill a Mockingbird', 'The Help', 'The Secret Life of Bees']
    },
    {
        'query': 'time travel history science fiction',
        'relevant_titles': ['The Time Machine', 'Outlander', '11/22/63']
    },
]


def precision_at_k(recommended: list, relevant: list, k: int) -> float:
    top_k = recommended[:k]
    hits  = sum(1 for t in top_k if any(rel.lower() in t.lower() for rel in relevant))
    return hits / k

def recall_at_k(recommended: list, relevant: list, k: int) -> float:
    top_k = recommended[:k]
    hits  = sum(1 for rel in relevant if any(rel.lower() in t.lower() for t in top_k))
    return hits / len(relevant) if relevant else 0

def reciprocal_rank(recommended: list, relevant: list) -> float:
    for i, t in enumerate(recommended, 1):
        if any(rel.lower() in t.lower() for rel in relevant):
            return 1.0 / i
    return 0.0


# ----- 5.2 Run evaluation -----
K_VALUES = [3, 5, 10]
results_log = []

print("🧪 Running evaluation on test cases...\n")

for tc in TEST_CASES:
    recs = hybrid_search(tc['query'], top_n=10)
    rec_titles = recs['title'].tolist()

    row = {'query': tc['query']}
    for k in K_VALUES:
        row[f'P@{k}']  = precision_at_k(rec_titles, tc['relevant_titles'], k)
        row[f'R@{k}']  = recall_at_k(rec_titles, tc['relevant_titles'], k)
    row['MRR'] = reciprocal_rank(rec_titles, tc['relevant_titles'])
    results_log.append(row)

    print(f"  Query: '{tc['query']}'")
    print(f"    Top 3 results: {rec_titles[:3]}")
    print(f"    P@5={row['P@5']:.2f}  R@5={row['R@5']:.2f}  MRR={row['MRR']:.2f}\n")


# ----- 5.3 Aggregate metrics -----
eval_df = pd.DataFrame(results_log)
print("\n📊 AVERAGE METRICS ACROSS ALL TEST CASES:")
print(eval_df[[f'P@{k}' for k in K_VALUES] + [f'R@{k}' for k in K_VALUES] + ['MRR']].mean().round(3).to_string())


# ----- 5.4 Coverage metric -----
all_recommended = set()
sample_queries = ['adventure', 'romance', 'mystery', 'science fiction',
                  'history', 'biography', 'fantasy', 'thriller', 'comedy', 'horror']
for q in sample_queries:
    recs = hybrid_search(q, top_n=20)
    all_recommended.update(recs['book_id'].tolist())

coverage = len(all_recommended) / len(df) * 100
print(f"\n📦 Catalogue Coverage: {coverage:.1f}% of books are reachable via recommendations")

In [ ]:
# ----- 5.5 Visualise evaluation metrics -----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('AI Recommendation Engine — Evaluation Metrics', fontsize=14, fontweight='bold')

# Precision and Recall bar chart
metrics_summary = eval_df[[f'P@{k}' for k in K_VALUES] + [f'R@{k}' for k in K_VALUES]].mean()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4']
metrics_summary.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Precision & Recall @K')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.4, label='0.5 baseline')
axes[0].legend()

# MRR per query
axes[1].bar(range(len(eval_df)), eval_df['MRR'], color='#4CAF50', edgecolor='white')
axes[1].set_xticks(range(len(eval_df)))
axes[1].set_xticklabels([q[:20]+'…' for q in eval_df['query']], rotation=15, ha='right', fontsize=8)
axes[1].set_title('Mean Reciprocal Rank (MRR) per Query')
axes[1].set_ylabel('MRR')
axes[1].set_ylim(0, 1)
axes[1].axhline(eval_df['MRR'].mean(), color='red', linestyle='--', label=f"Mean={eval_df['MRR'].mean():.2f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: evaluation_metrics.png")

---
# 🏆 STEP 6: Ranking & Filtering Logic
**Goal:** Apply a multi-factor ranking formula that combines semantic relevance score with book quality signals (rating, popularity, recency). Add UI-ready filter controls.

**Final ranking formula:**
```
final_rank = w1×similarity + w2×normalized_rating + w3×recency_score
```

In [ ]:
# ============================================================
# STEP 6: RANKING & FILTERING LOGIC
# ============================================================

# ----- 6.1 Precompute normalised rating and recency scores -----
df['norm_rating'] = (df['average_rating'] - df['average_rating'].min()) / \
                    (df['average_rating'].max() - df['average_rating'].min())

# Recency: scale publication year 0→1 (older=0, newer=1)
valid_years = df['original_publication_year'].replace(0, np.nan)
y_min = valid_years.min()
y_max = valid_years.max()
df['recency_score'] = (valid_years - y_min) / (y_max - y_min + 1e-9)
df['recency_score'] = df['recency_score'].fillna(0.5)  # Unknown year → neutral

print("✅ Precomputed: norm_rating, recency_score")


def advanced_recommend(
    query:          str,
    top_n:          int   = 10,
    alpha:          float = 0.6,   # SBERT vs TF-IDF blend
    w_similarity:   float = 0.7,   # Weight: AI similarity score
    w_rating:       float = 0.2,   # Weight: book rating
    w_recency:      float = 0.1,   # Weight: publication recency
    min_rating:     float = None,
    year_range:     tuple = None,
    author_filter:  str   = None,
    language_filter:str   = None,  # e.g. 'eng'
    sort_by:        str   = 'score',  # 'score', 'rating', 'year'
) -> pd.DataFrame:
    """
    Full-featured recommender with multi-factor ranking.
    Returns structured results ready for frontend display.
    """
    # Step 1: Get base hybrid scores
    base = hybrid_search(
        query, top_n=min(top_n * 5, len(df)),  # Over-fetch then re-rank
        alpha=alpha,
        min_rating=min_rating,
        year_range=year_range,
        author_hint=author_filter
    )

    # Step 2: Language filter
    if language_filter:
        base = base.merge(
            df[['book_id','language_code','norm_rating','recency_score']],
            on='book_id', how='left'
        )
        base = base[base['language_code'] == language_filter]
    else:
        base = base.merge(
            df[['book_id','norm_rating','recency_score']],
            on='book_id', how='left'
        )

    if base.empty:
        return pd.DataFrame(columns=['title','authors','average_rating','final_score'])

    # Step 3: Multi-factor ranking
    base['ranked_score'] = (
        w_similarity * base['final_score'] +
        w_rating     * base['norm_rating'] +
        w_recency    * base['recency_score']
    )

    # Step 4: Sort
    if sort_by == 'rating':
        base = base.sort_values('average_rating', ascending=False)
    elif sort_by == 'year':
        base = base.sort_values('original_publication_year', ascending=False)
    else:  # default: by ranked_score
        base = base.sort_values('ranked_score', ascending=False)

    return base.head(top_n)[[
        'book_id', 'title', 'authors', 'average_rating',
        'original_publication_year', 'image_url',
        'tfidf_score', 'sbert_score', 'final_score', 'ranked_score'
    ]].reset_index(drop=True)


# ----- 6.2 Demonstration -----
print("\n📚 DEMO: 'dark fantasy epic quest' — balanced ranking")
results = advanced_recommend(
    'dark fantasy epic quest',
    top_n=8,
    min_rating=3.8,
    w_similarity=0.7,
    w_rating=0.2,
    w_recency=0.1
)
print(results[['title', 'authors', 'average_rating', 'ranked_score']].to_string(index=False))

In [ ]:
# ----- 6.3 Diversity Re-ranking (MMR - Maximal Marginal Relevance) -----
# Prevents the same author / same series from dominating results

def mmr_rerank(results_df: pd.DataFrame, lambda_: float = 0.7, top_n: int = 5) -> pd.DataFrame:
    """
    Maximal Marginal Relevance: balance relevance vs diversity.
    lambda_=1.0 → pure relevance | lambda_=0.0 → pure diversity
    """
    if len(results_df) <= 1:
        return results_df

    book_ids   = results_df['book_id'].tolist()
    scores_map = dict(zip(results_df['book_id'], results_df['ranked_score']))

    # Get SBERT embeddings for this result set
    indices = [df[df['book_id'] == bid].index[0] for bid in book_ids if len(df[df['book_id']==bid])>0]
    embs    = book_embeddings[indices]

    selected  = [0]      # Start with highest-ranked book
    remaining = list(range(1, len(indices)))

    while len(selected) < min(top_n, len(indices)) and remaining:
        best_i, best_score = None, -1
        for i in remaining:
            rel  = scores_map.get(book_ids[i], 0)
            # Max similarity to already selected
            div  = max(cosine_similarity([embs[i]], [embs[s]])[0][0] for s in selected)
            mmr  = lambda_ * rel - (1 - lambda_) * div
            if mmr > best_score:
                best_score, best_i = mmr, i
        selected.append(best_i)
        remaining.remove(best_i)

    selected_ids = [book_ids[i] for i in selected]
    return results_df[results_df['book_id'].isin(selected_ids)].reset_index(drop=True)


# Demo with diversity
base_results = advanced_recommend('magic fantasy adventure', top_n=20)
diverse_results = mmr_rerank(base_results, lambda_=0.65, top_n=5)

print("✅ Diversity-ranked top 5 for 'magic fantasy adventure':")
print(diverse_results[['title', 'authors', 'average_rating']].to_string(index=False))

---
# 🔁 STEP 7: Feedback Data Handling
**Goal:** Collect user feedback (clicks, ratings, skips) and use it to:
1. Store feedback persistently (CSV / JSON)
2. Analyse feedback patterns
3. Incrementally update the ranking model without full retraining

**Feedback signals:**
| Signal | Weight |
|---|---|
| User clicked on book | +1.0 |
| User rated 5★ | +2.0 |
| User rated 4★ | +1.0 |
| User rated 3★ | 0.0 |
| User rated 1-2★ | -1.0 |
| User skipped / dismissed | -0.5 |

In [ ]:
# ============================================================
# STEP 7: FEEDBACK DATA HANDLING
# ============================================================
import json
import os
from datetime import datetime

FEEDBACK_FILE = 'user_feedback.json'
FEEDBACK_WEIGHTS = {
    'click':   1.0,
    'rate_5':  2.0,
    'rate_4':  1.0,
    'rate_3':  0.0,
    'rate_2': -1.0,
    'rate_1': -1.0,
    'skip':   -0.5,
}

# ----- 7.1 Feedback storage -----
def load_feedback() -> list:
    if os.path.exists(FEEDBACK_FILE):
        with open(FEEDBACK_FILE, 'r') as f:
            return json.load(f)
    return []

def save_feedback(feedback_log: list):
    with open(FEEDBACK_FILE, 'w') as f:
        json.dump(feedback_log, f, indent=2)

def record_feedback(
    user_id:   str,
    query:     str,
    book_id:   int,
    book_title:str,
    action:    str,   # 'click', 'rate_5', 'skip', etc.
):
    """
    Records one feedback event to disk.
    Call this whenever a user interacts with a recommendation.
    """
    fb = load_feedback()
    entry = {
        'timestamp':  datetime.now().isoformat(),
        'user_id':    user_id,
        'query':      query,
        'book_id':    book_id,
        'book_title': book_title,
        'action':     action,
        'weight':     FEEDBACK_WEIGHTS.get(action, 0),
    }
    fb.append(entry)
    save_feedback(fb)
    print(f"✅ Recorded: user={user_id} | '{book_title}' | {action}")


# ----- 7.2 Simulate some feedback events -----
print("🎭 Simulating user feedback events...\n")

sample_recs = advanced_recommend('adventure fantasy', top_n=5)

for i, row in sample_recs.iterrows():
    action = np.random.choice(['click','rate_5','rate_4','skip'],
                               p=[0.4, 0.2, 0.2, 0.2])
    record_feedback(
        user_id='user_001',
        query='adventure fantasy',
        book_id=row['book_id'],
        book_title=row['title'],
        action=action
    )

In [ ]:
# ----- 7.3 Feedback Analysis -----
fb_log = load_feedback()
fb_df  = pd.DataFrame(fb_log)

if not fb_df.empty:
    print(f"📋 Total feedback events: {len(fb_df)}")
    print(f"\n📊 Action distribution:")
    print(fb_df['action'].value_counts().to_string())

    print(f"\n🏆 Top books by feedback score:")
    book_scores = fb_df.groupby('book_title')['weight'].sum().sort_values(ascending=False)
    print(book_scores.head(10).to_string())

    print(f"\n🔍 Most queried topics:")
    print(fb_df['query'].value_counts().head(5).to_string())
else:
    print("No feedback collected yet.")

In [ ]:
# ----- 7.4 Feedback-Boosted Recommendations -----
# Re-rank results by adding accumulated user feedback scores

def feedback_boost_recommendations(
    query:        str,
    top_n:        int   = 10,
    boost_weight: float = 0.2,   # How much feedback influences ranking
) -> pd.DataFrame:
    """
    Returns recommendations boosted by historical user feedback.
    Books users liked before rank higher; disliked books rank lower.
    """
    # Get base recommendations
    recs = advanced_recommend(query, top_n=top_n * 3)

    # Load feedback
    fb_log = load_feedback()
    if not fb_log:
        return recs.head(top_n)

    fb_df = pd.DataFrame(fb_log)

    # Aggregate feedback per book_id
    book_fb = fb_df.groupby('book_id')['weight'].sum().reset_index()
    book_fb.columns = ['book_id', 'feedback_score']

    # Normalise feedback to [-1, 1]
    max_abs = book_fb['feedback_score'].abs().max()
    if max_abs > 0:
        book_fb['feedback_norm'] = book_fb['feedback_score'] / max_abs
    else:
        book_fb['feedback_norm'] = 0

    # Merge into recommendations
    recs = recs.merge(book_fb[['book_id','feedback_norm']], on='book_id', how='left')
    recs['feedback_norm'] = recs['feedback_norm'].fillna(0)

    # Boost the ranked score
    recs['boosted_score'] = recs['ranked_score'] + boost_weight * recs['feedback_norm']
    recs = recs.sort_values('boosted_score', ascending=False)

    return recs.head(top_n)[[
        'title', 'authors', 'average_rating',
        'ranked_score', 'feedback_norm', 'boosted_score'
    ]].reset_index(drop=True)


print("📚 Feedback-boosted results for 'adventure fantasy':")
print(feedback_boost_recommendations('adventure fantasy', top_n=5).to_string(index=False))

In [ ]:
# ----- 7.5 Periodic Model Update based on Feedback -----
# When enough feedback is collected, augment TF-IDF with feedback-weighted corpus

def retrain_with_feedback(min_feedback_events: int = 20):
    """
    Re-fits TF-IDF by up-weighting positively-rated books.
    Call this periodically (e.g. weekly) when feedback accumulates.
    """
    global tfidf_vectorizer, tfidf_matrix

    fb_log = load_feedback()
    if len(fb_log) < min_feedback_events:
        print(f"⏳ Not enough feedback yet ({len(fb_log)}/{min_feedback_events}). Skipping retrain.")
        return

    print(f"🔄 Retraining with {len(fb_log)} feedback events...")
    fb_df = pd.DataFrame(fb_log)

    # Aggregate feedback
    book_fb = fb_df.groupby('book_id')['weight'].sum().reset_index()

    # Build augmented content: repeat positively-rated book content
    augmented_content = df['content'].copy().tolist()

    for _, row in book_fb.iterrows():
        idx_list = df[df['book_id'] == row['book_id']].index.tolist()
        if idx_list:
            idx = idx_list[0]
            repeats = max(1, int(abs(row['weight'])))
            if row['weight'] > 0:
                # Positive feedback: add extra copies (boost visibility)
                augmented_content[idx] = (augmented_content[idx] + ' ') * repeats

    # Refit TF-IDF
    tfidf_vectorizer = TfidfVectorizer(
        max_features=15000, ngram_range=(1, 2),
        min_df=2, max_df=0.85, sublinear_tf=True
    )
    tfidf_matrix = tfidf_vectorizer.fit_transform(augmented_content)

    # Save updated models
    joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer_updated.pkl')
    joblib.dump(tfidf_matrix,     'tfidf_matrix_updated.pkl')

    print(f"✅ Retrained TF-IDF with feedback. New matrix: {tfidf_matrix.shape}")
    print("💾 Saved: tfidf_vectorizer_updated.pkl, tfidf_matrix_updated.pkl")


# Simulate retrain (will skip if < 20 events)
retrain_with_feedback(min_feedback_events=5)

---
# 🚀 STEP 8: End-to-End Demo
This final cell shows the complete pipeline from raw user query to final ranked recommendations — exactly as it would run when integrated into your web backend.

In [ ]:
# ============================================================
# STEP 8: COMPLETE END-TO-END PIPELINE
# ============================================================

def full_recommendation_pipeline(
    raw_query: str,
    user_id:   str = 'anonymous',
    top_n:     int = 10,
) -> dict:
    """
    Complete AI pipeline:
    1. Process query → extract intent & keywords
    2. Hybrid search (TF-IDF + SBERT)
    3. Multi-factor ranking
    4. Diversity re-ranking (MMR)
    5. Feedback boosting

    Returns JSON-serialisable dict ready for API response.
    """
    # Step 1: Query processing
    processed = process_query(raw_query)
    intent    = processed['intent']

    # Step 2 & 3: Ranked recommendations
    recs = advanced_recommend(
        query=raw_query,
        top_n=top_n * 3,
        min_rating=intent.get('min_rating'),
        year_range=intent.get('year_range'),
        author_filter=intent.get('author_hint'),
    )

    # Step 4: Diversity re-ranking
    diverse = mmr_rerank(recs, lambda_=0.65, top_n=top_n * 2)

    # Step 5: Feedback boost
    fb_log = load_feedback()
    if fb_log:
        fb_df     = pd.DataFrame(fb_log)
        book_fb   = fb_df.groupby('book_id')['weight'].sum().reset_index()
        book_fb.columns = ['book_id', 'fb_score']
        diverse   = diverse.merge(book_fb, on='book_id', how='left')
        diverse['fb_score'] = diverse['fb_score'].fillna(0)
        diverse['final'] = diverse['ranked_score'] + 0.15 * diverse['fb_score']
        diverse = diverse.sort_values('final', ascending=False)
    
    final_recs = diverse.head(top_n)

    # Format output
    output = {
        'query':        raw_query,
        'intent':       intent,
        'keywords':     [kw[0] for kw in processed['keywords']],
        'total_results':len(final_recs),
        'books': [
            {
                'rank':        i + 1,
                'book_id':     int(row['book_id']),
                'title':       row['title'],
                'authors':     row['authors'],
                'rating':      float(row['average_rating']),
                'year':        int(row['original_publication_year']),
                'image_url':   row.get('image_url', ''),
                'ai_score':    float(row['ranked_score'].round(4)),
            }
            for i, row in final_recs.iterrows()
        ]
    }
    return output


# ---- Run the full pipeline ----
result = full_recommendation_pipeline(
    raw_query='exciting adventure for young adults with magic',
    user_id='student_42',
    top_n=8
)

print(f"🔍 Query: {result['query']}")
print(f"🧩 Detected intent: {result['intent']}")
print(f"🔑 Keywords: {result['keywords']}")
print(f"📚 Results: {result['total_results']} books\n")
print(f"{'Rank':<5} {'Title':<45} {'Authors':<25} {'Rating':>6} {'AI Score':>9}")
print("-" * 95)
for b in result['books']:
    print(f"{b['rank']:<5} {b['title'][:44]:<45} {b['authors'][:24]:<25} {b['rating']:>6.2f} {b['ai_score']:>9.4f}")

print("\n✅ Pipeline complete. This JSON output is ready to send to your frontend.")

---
# 🌐 STEP 9: Save as Reusable Module (for Backend Integration)
Save the recommendation engine as a `.py` file that your Java/Python backend can call via subprocess or REST API.

In [ ]:
# ============================================================
# STEP 9: Export as standalone Python module
# ============================================================

module_code = '''
# recommend_engine.py
# Call from command line: python recommend_engine.py "your query here"

import sys, json
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained artifacts
df              = pd.read_csv("books_cleaned.csv")
tfidf_vec       = joblib.load("tfidf_vectorizer.pkl")
tfidf_mat       = joblib.load("tfidf_matrix.pkl")
book_embeddings = np.load("book_embeddings.npy")
sbert_model     = SentenceTransformer("all-MiniLM-L6-v2")

def recommend(query: str, top_n: int = 10):
    # SBERT scores
    q_emb    = sbert_model.encode([query], convert_to_numpy=True)
    s_scores = cosine_similarity(q_emb, book_embeddings).flatten()
    # TF-IDF scores
    q_vec    = tfidf_vec.transform([query])
    t_scores = cosine_similarity(q_vec, tfidf_mat).flatten()
    # Fuse
    fused    = 0.6 * s_scores + 0.4 * t_scores
    top_idx  = fused.argsort()[::-1][:top_n]
    results  = df.iloc[top_idx][["title","authors","average_rating","image_url"]].copy()
    results["score"] = fused[top_idx].round(4)
    return results.to_dict(orient="records")

if __name__ == "__main__":
    query = " ".join(sys.argv[1:]) if len(sys.argv) > 1 else "adventure"
    print(json.dumps(recommend(query)))
'''

with open('recommend_engine.py', 'w') as f:
    f.write(module_code)

print("💾 Saved: recommend_engine.py")
print("\n▶️  Test it from terminal:")
print("    python recommend_engine.py 'exciting fantasy adventure'")
print("\n▶️  Call from Java backend via ProcessBuilder:")
print("    Runtime.getRuntime().exec(new String[]{\"python3\", \"recommend_engine.py\", query})")

---
## 📁 Files Generated by This Notebook

| File | Description |
|---|---|
| `books_cleaned.csv` | Preprocessed book dataset |
| `tfidf_vectorizer.pkl` | Trained TF-IDF vectorizer |
| `tfidf_matrix.pkl` | TF-IDF sparse matrix for all books |
| `book_embeddings.npy` | Sentence-BERT embeddings (384-dim) |
| `user_feedback.json` | Collected user feedback events |
| `evaluation_metrics.png` | P@K, R@K, MRR charts |
| `recommend_engine.py` | Standalone module for backend integration |

---
## ⚡ Quick Reference — Run Order

```
First time:  Step 0 → 1 → 2 → 3A → 3B → 4 → 5 → 6 → 7 → 8 → 9
After that:  Step 3C (load saved) → then jump to any step
```